# FFM-OSLO v2.0 — CARLab 218 人 · 单耳 · GELU · FiLM

相对 v1 的四处改动：

| # | 改动 | 依据 |
|---|---|---|
| 1 | **单耳样本 + 右耳镜像** | 方差分解显示 v1 的 latent 有 **71.7%** 在编码"这是哪只耳"（PC1 与 side 相关 0.997）。镜像后所有样本进入同一坐标系，容量释放给个体差异。样本数 218 → **436** |
| 2 | **按受试者划分 train/val** | 同一人左右耳镜像后相关 **0.978**，按样本随机划分 = 数据泄漏 |
| 3 | **ReLU → GELU** | 与 fc 层一致；HRTF 是双极信号，ReLU 截断负半轴 |
| 5 | **频率硬截断 16 kHz** | 源文件 `hrirfilt` 在 16.3 kHz 被数字滤波器切断（-418 dB/oct），其上谱粗糙度 1.0 → 6.5 dB = 阻带噪声。只掩码损失不够，噪声仍会进 encoder / 谱梯度损失 / FiLM。改为**载入时截断**，129 → **93 bins** |
| 4 | **加入 per-dataset FiLM** | 为将来并入 HUTUBS / SONICOM 预留。**只按频率、不按方向** —— 见 Cell 2 的说明 |

数据：`CARLab_left/` + `CARLab_right/`，218 人，raw 管线，48 kHz，HEALPix nside=16。
建议训练频段 **500 Hz – 16 kHz**（<500 Hz 实测不可靠，>16 kHz 原始数据滚降到 −42 dB）。

In [ ]:
# ==========================================================================
# Cell 0 — 数据准备
# ==========================================================================
import os, zipfile
from google.colab import drive

drive.mount('/content/drive')

ZIPS = [
    '/content/drive/MyDrive/ELEC5021/CARLab/CARLab_44k1_left.zip',
    '/content/drive/MyDrive/ELEC5021/CARLab/CARLab_44k1_right.zip',
]
EXTRACT_DIR = '/content/dataset/'
os.makedirs(EXTRACT_DIR, exist_ok=True)

for zp in ZIPS:
    if not os.path.exists(zp):
        print(f'[缺失] {zp}'); continue
    with zipfile.ZipFile(zp) as z:
        z.extractall(EXTRACT_DIR)
    folder = os.path.join(EXTRACT_DIR, os.path.basename(zp).replace('.zip', ''))
    n = len([f for f in os.listdir(folder) if f.endswith('.npy')]) if os.path.isdir(folder) else 0
    print(f'{os.path.basename(zp)} -> {folder}  ({n} npy)')

# 左右镜像置换（由 analysis/finalize_carlab.py 生成，已验证是精确双射）
MIRROR_SRC = '/content/drive/MyDrive/healpix_mirror_idx.npy'
print('\nmirror idx:', '找到' if os.path.exists(MIRROR_SRC) else '★缺失，见下一 cell 现场生成')

In [ ]:
!pip install healpy -q

In [ ]:
# ==========================================================================
# Cell 2b — ★ 采样率与频率轴一致性检查（并入多个数据集前必跑）
#
# 为什么需要：同一个 bin 索引在不同采样率下对应【不同的物理频率】。
#   48   kHz / 256 : df = 187.50 Hz,  bin 64 = 12000 Hz
#   44.1 kHz / 256 : df = 172.27 Hz,  bin 64 = 11025 Hz
# 在 notch 带错位接近 1 kHz —— 和一个 notch 本身的宽度同量级。
# 混着训 = 把最关心的特征抹平，而且不会报错，只会悄悄变差。
# ==========================================================================
import os, json
import numpy as np

# 把要一起训练的所有数据目录列在这里
DATA_DIRS = {
    'CARLab-L': '/content/dataset/CARLab_44k1_left',
    'CARLab-R': '/content/dataset/CARLab_44k1_right',
    # 'HUTUBS-L':  '/content/dataset/HUTUBS_left',
    # 'SONICOM-L': '/content/dataset/SONICOM_left',
}

EXPECT_FS   = 44100      # 目标采样率
EXPECT_NFFT = 256
EXPECT_BINS = 129
EXPECT_NPIX = 3072

print(f'目标: fs={EXPECT_FS} Hz, nfft={EXPECT_NFFT}, {EXPECT_BINS} bins, '
      f'df={EXPECT_FS/EXPECT_NFFT:.3f} Hz, fmax={EXPECT_FS/2:.0f} Hz')
print('=' * 78)

ok = True
for tag, d in DATA_DIRS.items():
    if not os.path.isdir(d):
        print(f'  {tag:<12} ★ 目录不存在: {d}'); ok = False; continue

    # ---- 1) 声明的元数据 ----
    mp = os.path.join(d, 'meta.json')
    if os.path.exists(mp):
        m = json.load(open(mp))
        fs, nfft, nb = m.get('sample_rate'), m.get('nfft'), m.get('n_bins')
        src = 'meta.json'
    else:
        fs = nfft = nb = None
        src = '★ 无 meta.json'

    # ---- 2) 实际数据的形状 ----
    files = sorted(f for f in os.listdir(d) if f.endswith('.npy'))
    x = np.load(os.path.join(d, files[0]))
    shape_ok = (x.shape == (EXPECT_BINS, EXPECT_NPIX))

    bad = []
    if fs != EXPECT_FS:      bad.append(f'fs={fs}')
    if nfft != EXPECT_NFFT:  bad.append(f'nfft={nfft}')
    if nb != EXPECT_BINS:    bad.append(f'n_bins={nb}')
    if not shape_ok:         bad.append(f'shape={x.shape}')
    ok &= not bad

    print(f"  {tag:<12} {len(files):>4} 文件 | {src:<12} | fs={fs} nfft={nfft} "
          f"| shape={x.shape} | {'OK' if not bad else '★ ' + ', '.join(bad)}")

print('=' * 78)

# ---- 3) 不依赖元数据的物理核对：高频滚降拐点 ----
# CARLab 的原始测量在 ~16 kHz 以上急剧滚降（测量系统上限）。
# 找到拐点所在的 bin，就能反推 df —— 与声明的采样率互相印证。
d0 = DATA_DIRS['CARLab-L']
if os.path.isdir(d0):
    fs_arr = sorted(f for f in os.listdir(d0) if f.endswith('.npy'))[:20]
    A = np.stack([np.load(os.path.join(d0, f)) for f in fs_arr]).mean(axis=(0, 2))
    hi = np.arange(len(A)) > 0.4 * len(A)
    knee = int(np.argmin(np.gradient(A) * hi))          # 下降最陡处
    for fs_try in (44100, 48000, 96000):
        print(f'  若 fs={fs_try:>5}: 滚降拐点落在 {knee * fs_try / EXPECT_NFFT / 1000:5.1f} kHz'
              f"{'   <- 与 CARLab 的 ~16 kHz 测量上限吻合' if abs(knee*fs_try/EXPECT_NFFT-16000) < 2500 else ''}")

print('\n' + ('全部一致 ✔' if ok else '★ 有不一致项，先修好再训练'))
assert ok, '数据集配置不一致'

# ---- 4) ★ 频率上限落点 ----
FREQ_CAP_HZ = 16000
DF_ = EXPECT_FS / EXPECT_NFFT
N_KEEP = int(np.floor(FREQ_CAP_HZ / DF_)) + 1
print(f'\n频率上限 {FREQ_CAP_HZ} Hz  ->  保留 bins 0..{N_KEEP-1} '
      f'({N_KEEP}/{EXPECT_BINS}), 实际 fmax = {(N_KEEP-1)*DF_:.0f} Hz')
print(f'  丢弃 {EXPECT_BINS-N_KEEP} 个 bin ({N_KEEP*DF_:.0f}-{(EXPECT_BINS-1)*DF_:.0f} Hz)'
      ' —— 该段 CARLab 为阻带噪声, HUTUBS/SONICOM 虽有数据但多库训练取最小上限')


In [ ]:
# ==========================================================================
# Cell 2 — 模型定义（GELU + FiLM）
# ==========================================================================
import numpy as np
import torch
import torch.nn as nn
import healpy as hp


# ---------------------------------------------- 球面卷积
class OSLOConv(nn.Module):
    """HEALPix 上的各向异性卷积：每个像素 + 8 邻居，9 个独立权重。"""
    def __init__(self, in_channels, out_channels, nside=16):
        super().__init__()
        self.npix = hp.nside2npix(nside)
        neighbors = hp.get_all_neighbours(nside, np.arange(self.npix))
        rf = np.concatenate([neighbors, np.arange(self.npix)[np.newaxis, :]], axis=0)
        rf[rf == -1] = self.npix
        self.register_buffer('neighborhood', torch.tensor(rf, dtype=torch.int64))
        self.weight = nn.Parameter(torch.empty(out_channels, in_channels, 9))
        self.bias = nn.Parameter(torch.empty(out_channels))
        nn.init.kaiming_uniform_(self.weight, a=np.sqrt(5))
        nn.init.zeros_(self.bias)

    def forward(self, x):
        B, C, _ = x.shape
        pad = torch.zeros(B, C, 1, device=x.device, dtype=x.dtype)
        g = torch.cat([x, pad], dim=2)[:, :, self.neighborhood]
        return torch.einsum('bcqn,ocq->bon', g, self.weight) + self.bias.view(1, -1, 1)


class OSLOResBlock(nn.Module):
    """频率隔离的残差块，处理 (B*F, C, N)。
    ★ v2: ReLU -> GELU。HRTF 的 log-magnitude 是双极量（负值表示衰减），
      ReLU 会把负半轴直接截断；GELU 平滑且保留负值，也与 fc 层一致。"""
    def __init__(self, in_channels, out_channels, nside=16, num_groups=8):
        super().__init__()
        self.shortcut = (nn.Conv1d(in_channels, out_channels, 1)
                         if in_channels != out_channels else nn.Identity())
        self.conv1 = OSLOConv(in_channels, out_channels, nside)
        self.gn1 = nn.GroupNorm(num_groups, out_channels)
        self.conv2 = OSLOConv(out_channels, out_channels, nside)
        self.gn2 = nn.GroupNorm(num_groups, out_channels)
        self.act = nn.GELU()

    def forward(self, x):
        res = self.shortcut(x)
        out = self.act(self.gn1(self.conv1(x)))
        out = self.gn2(self.conv2(out))
        return self.act(out + res)


# ---------------------------------------------- per-dataset FiLM
class DatasetFiLM(nn.Module):
    """为每个数据集学一条【逐频率】的 scale/shift 曲线。

    ★★ 关键设计约束：只按频率，【不】按方向。

      per-frequency 仿射 = 一条 EQ 曲线（129 个自由度/数据集）。它能吸收的只有
      "整个数据集共有的、与方向无关的染色" —— 也就是扬声器/麦克风响应、
      均衡约定、整体电平。它在结构上【无法】表达方向相关的差异，
      因此人群解剖差异（不同库的耳朵大小分布不同）仍然必须留在 z 里。

      对比：按 (频率 x 方向) 减去每库均值 = 3072x129 个自由度，
      足以把真实的人群结构一并扣掉 —— 那正是被指出的 wrong direction。

      所以 FiLM 与"减均值"不是同一件事的两种写法，它们的表达能力差了 3072 倍。

    num_datasets=1 时是恒等映射（只学到一条全局 EQ，无害），
    将来并入 HUTUBS / SONICOM 时把 num_datasets 调大即可。
    """
    def __init__(self, num_datasets, num_bins):
        super().__init__()
        self.log_gamma = nn.Parameter(torch.zeros(num_datasets, num_bins))  # exp(0)=1
        self.beta = nn.Parameter(torch.zeros(num_datasets, num_bins))

    def forward(self, x, dataset_id):
        """x: (B, F, N)   dataset_id: (B,) long"""
        g = torch.exp(self.log_gamma[dataset_id]).unsqueeze(-1)   # (B,F,1)
        b = self.beta[dataset_id].unsqueeze(-1)
        return x * g + b


# ---------------------------------------------- 编码器 / 解码器
class Isolated_OSLO_Encoder(nn.Module):
    """每个频点独立编码 -> latent (B, F, latent_dim)"""
    def __init__(self, nside=16, latent_dim_per_freq=64):
        super().__init__()
        self.npix = hp.nside2npix(nside)
        self.latent_dim = latent_dim_per_freq
        self.enc1 = OSLOResBlock(1, 16, nside)
        self.enc2 = OSLOResBlock(16, 32, nside)
        self.fc_encode = nn.Sequential(
            nn.Flatten(), nn.Linear(32 * self.npix, 512), nn.GELU(),
            nn.Linear(512, self.latent_dim))

    def forward(self, x):
        B, F, N = x.shape
        z = self.fc_encode(self.enc2(self.enc1(x.reshape(B * F, 1, N))))
        return z.reshape(B, F, self.latent_dim)


class Conditioned_OSLO_Decoder(nn.Module):
    """FFM 频率条件化解码器 + 输出端 per-dataset FiLM"""
    def __init__(self, nside=16, latent_dim_per_freq=64, freq_embed_dim=32,
                 num_datasets=1, num_bins=129):
        super().__init__()
        self.npix = hp.nside2npix(nside)
        self.latent_dim = latent_dim_per_freq
        self.freq_embed_dim = freq_embed_dim
        self.ffm_B = nn.Parameter(torch.randn(freq_embed_dim // 2))
        self.fc_decode = nn.Sequential(
            nn.Linear(self.latent_dim + freq_embed_dim, 512), nn.GELU(),
            nn.Linear(512, 32 * self.npix))
        self.dec1 = OSLOResBlock(32, 16, nside)
        self.dec_final = OSLOConv(16, 1, nside)
        self.film = DatasetFiLM(num_datasets, num_bins)

    def get_freq_embedding(self, num_freqs, device):
        f = torch.arange(num_freqs, device=device).float() / (num_freqs - 1)
        proj = 2 * np.pi * f.unsqueeze(1) * self.ffm_B.unsqueeze(0)
        return torch.cat([torch.sin(proj), torch.cos(proj)], dim=1)

    def forward(self, z, dataset_id=None):
        B, F, _ = z.shape
        fe = self.get_freq_embedding(F, z.device).unsqueeze(0).expand(B, -1, -1)
        h = torch.cat([z, fe], dim=-1).reshape(B * F, -1)
        h = self.fc_decode(h).reshape(B * F, 32, self.npix)
        out = self.dec_final(self.dec1(h)).reshape(B, F, self.npix)
        if dataset_id is not None:
            out = self.film(out, dataset_id)
        return out


class FFM_OSLO_Autoencoder(nn.Module):
    def __init__(self, nside=16, latent_dim_per_freq=64, freq_embed_dim=32,
                 num_datasets=1, num_bins=129):
        super().__init__()
        self.encoder = Isolated_OSLO_Encoder(nside, latent_dim_per_freq)
        self.decoder = Conditioned_OSLO_Decoder(nside, latent_dim_per_freq,
                                                freq_embed_dim, num_datasets, num_bins)

    def forward(self, x, dataset_id=None):
        z = self.encoder(x)
        return self.decoder(z, dataset_id), z


print('模型定义完成 (GELU + FiLM)')

In [ ]:
# ==========================================================================
# Cell 3 — 配置 · 单耳数据集（右耳镜像）· 按受试者划分
# ==========================================================================
import os, re, random
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import healpy as hp

config = {
    'model_type': 'FFM-OSLO-v2-CARLab',
    'nside': 16, 'latent_dim': 64, 'freq_embed_dim': 32,
    'sample_rate': 44100,           # 与 HUTUBS/SONICOM 对齐
    'num_bins_raw': 129,            # npy 文件里的原始 bin 数 (0 ~ fs/2)
    'freq_cap_hz': 16000,           # ★ 硬上限, 见下方 cap 说明
    'num_datasets': 1,              # 并入 HUTUBS/SONICOM 时改这里
    'lr': 1e-4, 'batch_size': 8, 'weight_decay': 1e-4,
    'w_mse': 1.0, 'w_spectral': 0.1,
    'band_lo_hz': 500,              # 低端仅做损失掩码(<500 Hz 数据本身是真的)
    'save_dir': '/content/drive/MyDrive/HRTF_Checkpoints',
    'left_dir': '/content/dataset/CARLab_44k1_left',
    'right_dir': '/content/dataset/CARLab_44k1_right',
    'mirror_idx': '/content/drive/MyDrive/healpix_mirror_idx.npy',
}

# ---------------------------- ★ 频率硬上限 ----------------------------
# 为什么是"硬截断"而不是只做损失掩码：
#   CARLab 的源文件 hrirfilt 在 16.3 kHz 处被数字滤波器切断（斜率 -418 dB/oct,
#   拐点跨受试者 std 仅 94 Hz = 固定的数字处理）。该点以上谱粗糙度从
#   1.0 dB 跳到 6.5-7.3 dB —— 阻带噪声的特征, 不是信号。
# 只把损失掩码掉是不够的, 因为那段噪声仍然会:
#   (1) 经 encoder 进入 z, 占掉本就紧张的 latent 容量(方差分解已显示容量不足);
#   (2) 进入 SpectralGradientLoss —— 它对【全部】bin 求一阶差分, 没有频段掩码,
#       噪声的差分幅度是信号的 6 倍, w_spectral=0.1 下这不是小量;
#   (3) 污染 FiLM 的高频参数与 mean-HRTF 基线。
# 所以在【载入时】就切掉, 模型永远看不到那段。
DF = config['sample_rate'] / 256.0                        # 172.266 Hz
N_BINS = int(np.floor(config['freq_cap_hz'] / DF)) + 1    # 含 bin 0 -> 93
config['num_bins'] = N_BINS
config['band_hi_hz'] = int(round((N_BINS - 1) * DF))       # 15848, 供绘图 cell 使用
print(f"频率上限 {config['freq_cap_hz']} Hz -> 保留 {N_BINS}/{config['num_bins_raw']} bins, "
      f"实际 fmax {config['band_hi_hz']} Hz (df {DF:.3f} Hz)")

resume_training   = False
fresh_epochs      = 150
additional_epochs = 50

os.makedirs(config['save_dir'], exist_ok=True)
latest_path = os.path.join(config['save_dir'], 'FFM2_latest.pth')
best_path   = os.path.join(config['save_dir'], 'FFM2_best.pth')

seed = 4
torch.manual_seed(seed); np.random.seed(seed); random.seed(seed)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(seed)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
torch.backends.cudnn.benchmark = True


# ------------------------------ 镜像置换 ------------------------------
if os.path.exists(config['mirror_idx']):
    MIRROR = np.load(config['mirror_idx'])
else:                                    # 现场生成（与 finalize_carlab.py 一致）
    npix = hp.nside2npix(config['nside'])
    th, ph = hp.pix2ang(config['nside'], np.arange(npix))
    MIRROR = hp.ang2pix(config['nside'], th, (-ph) % (2 * np.pi))
assert len(np.unique(MIRROR)) == len(MIRROR), '镜像不是双射'
assert np.array_equal(MIRROR[MIRROR], np.arange(len(MIRROR))), '镜像不是对合'
print(f'镜像置换 OK (精确双射且对合)')


# ------------------------------ 单耳数据集 ------------------------------
class MonauralHRTFDataset(Dataset):
    """每个样本 = 一只耳朵。右耳沿中矢面镜像，使全部样本进入"左耳坐标系"。

    为什么必须镜像：v1 的方差分解显示 71.7% 的 latent 方差在编码 side —— 一个
    我们本来就知道的比特。镜像后这个自由度消失，容量全部留给个体差异。
    """
    def __init__(self, left_dir, right_dir, subject_ids, mirror, dataset_id=0):
        self.items, self.mirror, self.dataset_id = [], mirror, dataset_id
        for sid in sorted(subject_ids):
            l = os.path.join(left_dir, f'{sid}_left_full.npy')
            r = os.path.join(right_dir, f'{sid}_right_full.npy')
            if os.path.exists(l): self.items.append((sid, 'left', l))
            if os.path.exists(r): self.items.append((sid, 'right', r))

    def __len__(self):
        return len(self.items)

    def __getitem__(self, i):
        sid, side, path = self.items[i]
        x = np.load(path).astype(np.float32)[:N_BINS]   # (129,3072) -> (N_BINS,3072)
        if side == 'right':
            x = x[:, self.mirror]                     # -> 左耳坐标系
        return torch.from_numpy(x), torch.tensor(self.dataset_id), f'{sid}_{side}'


# --------------------- ★ 按【受试者】划分，不是按样本 ---------------------
# 同一人的左右耳镜像后相关 0.978：若按样本随机切分，某人左耳进 train、
# 右耳进 val 是大概率事件 —— 那是标准的数据泄漏，val 会假性变好。
all_ids = sorted({f.split('_')[0] for f in os.listdir(config['left_dir'])
                  if f.endswith('.npy')})
rng = np.random.default_rng(seed)
perm = list(all_ids); rng.shuffle(perm)
n_tr = int(0.8 * len(perm))
train_ids, val_ids = perm[:n_tr], perm[n_tr:]
assert not (set(train_ids) & set(val_ids))

train_dataset = MonauralHRTFDataset(config['left_dir'], config['right_dir'], train_ids, MIRROR)
val_dataset   = MonauralHRTFDataset(config['left_dir'], config['right_dir'], val_ids,   MIRROR)
print(f'受试者 {len(all_ids)}  ->  train {len(train_ids)} 人 / {len(train_dataset)} 耳'
      f'   |   val {len(val_ids)} 人 / {len(val_dataset)} 耳')

g = torch.Generator(); g.manual_seed(seed)
train_loader = DataLoader(train_dataset, batch_size=config['batch_size'], shuffle=True, generator=g)
val_loader   = DataLoader(val_dataset,   batch_size=config['batch_size'], shuffle=False)


# ------------------------------ 频段掩码 ------------------------------
freqs = np.arange(N_BINS) * DF                        # 0 ~ band_hi_hz
band = torch.tensor(freqs >= config['band_lo_hz'])    # 上限已由硬截断保证
print(f"损失频段 {config['band_lo_hz']}-{config['band_hi_hz']} Hz -> "
      f'{int(band.sum())}/{len(band)} 个频点')


# ------------------------------ 损失 ------------------------------
class SpectralGradientLoss(nn.Module):
    """沿频率轴的一阶差分 MSE —— 鼓励模型贴合 notch 的陡峭形状"""
    def forward(self, pred, target):
        return F.mse_loss(pred[:, 1:, :] - pred[:, :-1, :],
                          target[:, 1:, :] - target[:, :-1, :])


# ------------------------------ 模型 ------------------------------
model = FFM_OSLO_Autoencoder(
    nside=config['nside'], latent_dim_per_freq=config['latent_dim'],
    freq_embed_dim=config['freq_embed_dim'],
    num_datasets=config['num_datasets'], num_bins=config['num_bins']).to(device)

optimizer = optim.Adam(model.parameters(), lr=config['lr'], weight_decay=config['weight_decay'])
criterion_mse, criterion_mae = nn.MSELoss(), nn.L1Loss()
criterion_spectral = SpectralGradientLoss().to(device)

model.eval()
with torch.no_grad():
    d = torch.randn(2, N_BINS, 3072).to(device)
    o, z = model(d, torch.zeros(2, dtype=torch.long, device=device))
    print(f'输入 {tuple(d.shape)} | 输出 {tuple(o.shape)} | latent {tuple(z.shape)}')
    assert o.shape == d.shape


# ------------------------------ mean-HRTF 基线 ------------------------------
# ★ 新数据分布和 v1 不同，旧的 2.14 / 2.33 dB 不可比。先算一个本数据集的参照。
with torch.no_grad():
    tr = torch.stack([train_dataset[i][0] for i in range(len(train_dataset))])
    mu = tr.mean(0, keepdim=True)
    va = torch.stack([val_dataset[i][0] for i in range(len(val_dataset))])
    bl_all  = torch.sqrt(((va - mu) ** 2).mean()).item()
    bl_band = torch.sqrt(((va - mu)[:, band, :] ** 2).mean()).item()
print(f'\nmean-HRTF 基线 (val): 全频段 {bl_all:.3f} dB | '
      f"{config['band_lo_hz']}-{config['band_hi_hz']}Hz {bl_band:.3f} dB")
del tr, va

In [ ]:
# ==========================================================================
# Cell 4 — 训练
# ==========================================================================
history = {'train_rmse': [], 'val_rmse': [], 'val_mae': [], 'train_loss_total': []}
best_val_rmse, done_epochs = float('inf'), 0
band_dev = band.to(device)

if resume_training:
    if not os.path.exists(latest_path):
        raise FileNotFoundError(f'开启了续训但找不到存档: {latest_path}\n第一次训练请把 resume_training 改为 False')
    ck = torch.load(latest_path, map_location=device, weights_only=False)
    model.load_state_dict(ck['model_state_dict'])
    optimizer.load_state_dict(ck['optimizer_state_dict'])
    history = ck.get('history', history)
    best_val_rmse = ck.get('val_rmse', float('inf'))
    done_epochs = ck.get('epoch', -1) + 1
    for pg in optimizer.param_groups:
        pg['lr'] = config['lr']                      # 热重启
    epochs = additional_epochs
    print(f'从第 {done_epochs} 轮恢复，历史最佳 {best_val_rmse:.4f}，本次再跑 {epochs} 轮')
else:
    epochs = fresh_epochs
    print(f'从零开始训练 {epochs} 轮')

scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs, eta_min=1e-6)


def masked_rmse(pred, target):
    return torch.sqrt(((pred - target)[:, band_dev, :] ** 2).mean())


for ep in range(epochs):
    # ---------------- train ----------------
    model.train()
    tot, mse_sum, nb = 0.0, 0.0, 0
    for x, did, _ in train_loader:
        x, did = x.to(device), did.to(device)
        optimizer.zero_grad()
        rec, _ = model(x, did)
        # 只在有效频段上算损失
        loss_mse = criterion_mse(rec[:, band_dev, :], x[:, band_dev, :])
        loss_spe = criterion_spectral(rec, x)
        loss = config['w_mse'] * loss_mse + config['w_spectral'] * loss_spe
        loss.backward(); optimizer.step()
        tot += loss.item(); mse_sum += loss_mse.item(); nb += 1
    scheduler.step()

    # ---------------- val ----------------
    model.eval()
    v_mse, v_mae, vb = 0.0, 0.0, 0
    with torch.no_grad():
        for x, did, _ in val_loader:
            x, did = x.to(device), did.to(device)
            rec, _ = model(x, did)
            v_mse += criterion_mse(rec[:, band_dev, :], x[:, band_dev, :]).item()
            v_mae += criterion_mae(rec[:, band_dev, :], x[:, band_dev, :]).item()
            vb += 1

    tr_rmse, va_rmse, va_mae = (mse_sum / nb) ** .5, (v_mse / vb) ** .5, v_mae / vb
    history['train_loss_total'].append(tot / nb)
    history['train_rmse'].append(tr_rmse)
    history['val_rmse'].append(va_rmse)
    history['val_mae'].append(va_mae)

    gap = bl_band - va_rmse
    print(f'Epoch {done_epochs+ep+1:>3} | train {tr_rmse:.4f} | val {va_rmse:.4f} dB '
          f'| MAE {va_mae:.4f} | vs baseline {gap:+.3f} dB | lr {scheduler.get_last_lr()[0]:.2e}')

    ck = {'epoch': done_epochs + ep, 'model_state_dict': model.state_dict(),
          'optimizer_state_dict': optimizer.state_dict(), 'history': history,
          'val_rmse': va_rmse, 'config': config,
          'train_ids': train_ids, 'val_ids': val_ids}
    torch.save(ck, latest_path)
    if va_rmse < best_val_rmse:
        best_val_rmse = va_rmse
        torch.save(ck, best_path)
        print(f'    ↳ 新最佳，已写入 {os.path.basename(best_path)}')

print(f'\n训练结束。最佳 val RMSE = {best_val_rmse:.4f} dB '
      f'(mean-HRTF 基线 {bl_band:.3f} dB)')

In [ ]:
# ==========================================================================
# Cell 5 — 载入 checkpoint（只做评估时从这里开始，跳过 Cell 4）
# ==========================================================================
ck = torch.load(best_path, map_location=device, weights_only=False)

# ★ v2.1 起模型是 93 bin。v2.0 及更早的存档 FiLM 是 129 bin, 无法直接载入。
ck_bins = ck['model_state_dict']['decoder.film.beta'].shape[1]
assert ck_bins == config['num_bins'], (
    f"存档是 {ck_bins} bin, 当前模型是 {config['num_bins']} bin —— "
    '频率上限改过之后旧存档不可用, 需要重新训练 (或把 freq_cap_hz 改回去)')

model.load_state_dict(ck['model_state_dict'])
model.eval()
history = ck.get('history', {})
print(f"载入 {os.path.basename(best_path)} | epoch {ck['epoch']} | val RMSE {ck['val_rmse']:.4f} dB")

# 存档里带了划分，确保评估用的是同一批人
if 'val_ids' in ck:
    val_ids = ck['val_ids']; train_ids = ck['train_ids']
    val_dataset = MonauralHRTFDataset(config['left_dir'], config['right_dir'], val_ids, MIRROR)
    print(f'val: {len(val_ids)} 人 / {len(val_dataset)} 耳')

In [ ]:
# ==========================================================================
# Cell 6 — 训练曲线
# ==========================================================================
import matplotlib.pyplot as plt

ep = np.arange(1, len(history['val_rmse']) + 1)
fig, ax = plt.subplots(1, 2, figsize=(14, 4.5), dpi=120)

ax[0].plot(ep, history['train_rmse'], label='train RMSE', lw=2)
ax[0].plot(ep, history['val_rmse'], label='val RMSE', lw=2)
ax[0].axhline(bl_band, color='gray', ls='--', lw=1.5, label=f'mean-HRTF baseline ({bl_band:.2f} dB)')
ax[0].set_xlabel('epoch'); ax[0].set_ylabel('RMSE (dB)')
ax[0].set_title(f"{config['model_type']}  ({config['band_lo_hz']}-{config['band_hi_hz']} Hz)")
ax[0].legend(); ax[0].grid(alpha=.3)

ax[1].semilogy(ep, history['train_loss_total'], lw=2, label='train total loss')
ax[1].plot(ep, history['val_mae'], lw=2, label='val MAE (dB)')
ax[1].set_xlabel('epoch'); ax[1].legend(); ax[1].grid(alpha=.3)
ax[1].set_title('loss / MAE')

plt.tight_layout(); plt.show()

best_ep = int(np.argmin(history['val_rmse'])) + 1
print(f"最佳 epoch {best_ep}: val RMSE {min(history['val_rmse']):.4f} dB "
      f'(比基线好 {bl_band - min(history["val_rmse"]):.3f} dB)')

In [ ]:
# ==========================================================================
# Cell 7 — 频谱重建对比（5 个代表方向，左右耳）
# ==========================================================================
import matplotlib.pyplot as plt
import healpy as hp

# ---- 从 val 里挑一个人，取他的左右耳 ----
val_subjects = sorted({it[0] for it in val_dataset.items})
TARGET = val_subjects[0]          # 想看别人就改这里
print(f'受试者 {TARGET}   (val 共 {len(val_subjects)} 人: {val_subjects[:6]} ...)')


def predict_ear(sid, side):
    """返回 (gt, pred)，都已还原到【该耳自己的原始坐标系】。
    模型工作在左耳坐标系；右耳的输入被镜像过，所以输出要再镜像一次还原
    （镜像是对合，M(M(x)) = x）。"""
    for s, sd, path in val_dataset.items:
        if s == sid and sd == side:
            x = np.load(path).astype(np.float32)[:N_BINS]
            xin = x[:, MIRROR] if side == 'right' else x
            with torch.no_grad():
                p, _ = model(torch.from_numpy(xin)[None].to(device),
                             torch.zeros(1, dtype=torch.long, device=device))
            p = p[0].cpu().numpy()
            if side == 'right':
                p = p[:, MIRROR]                    # 还原
            return x, p
    return None, None


gt_l, pred_l = predict_ear(TARGET, 'left')
gt_r, pred_r = predict_ear(TARGET, 'right')

directions = {
    'Front (az 0, el 0)':            (np.pi / 2, 0.0),
    'Left (az 90, el 0)':            (np.pi / 2, np.pi / 2),
    'Right (az -90, el 0)':          (np.pi / 2, -np.pi / 2),
    'Left upper (az 45, el 45)':     (np.pi / 4, np.pi / 4),
    'Right upper (az -45, el 45)':   (np.pi / 4, -np.pi / 4),
}
freqs_khz = freqs / 1000.0

fig, axes = plt.subplots(len(directions), 1, figsize=(13, 4 * len(directions)), dpi=120)
for ax, (name, (th, ph)) in zip(axes, directions.items()):
    pix = hp.ang2pix(config['nside'], th, ph)
    ax.plot(freqs_khz, gt_l[:, pix],   color='#1f77b4', lw=2.2, label='Left  GT')
    ax.plot(freqs_khz, pred_l[:, pix], color='#00BFFF', lw=2.0, ls='--', label='Left  Pred')
    if gt_r is not None:
        ax.plot(freqs_khz, gt_r[:, pix],   color='#d62728', lw=2.2, label='Right GT')
        ax.plot(freqs_khz, pred_r[:, pix], color='#ff7f0e', lw=2.0, ls='--', label='Right Pred')
    ax.axvspan(6, 12, color='orange', alpha=.10)                       # notch 带
    ax.axvspan(0, config['band_lo_hz'] / 1000, color='gray', alpha=.12)
    ax.set_xlim(0, freqs_khz[-1]); ax.set_xlabel('frequency (kHz)')
    ax.set_ylabel('magnitude (dB)'); ax.set_title(f'{TARGET} — {name}')
    ax.legend(fontsize=8, ncol=2); ax.grid(alpha=.3)
plt.tight_layout(); plt.show()
print(f"横轴已在 {config['band_hi_hz']} Hz 截断（数据在此以上是噪声）；"
      '灰色 = <500 Hz 不计损失；橙色 = 6-12 kHz notch 带')

In [ ]:
# ==========================================================================
# Cell 8 — HEALPix 空间热力图（GT / Pred / 误差）
# ==========================================================================
import matplotlib.pyplot as plt
import healpy as hp


def plot_healpix_map(gt, pred, target_hz, subject, ear='Left'):
    # ★ v1 这里用 16000/128 算频率索引是错的：bin 步进应为 fs/256。
    df = config['sample_rate'] / 256.0
    k = int(round(target_hz / df))
    k = min(max(k, 0), config['num_bins'] - 1)
    f_actual = k * df

    g, p = gt[k, :], pred[k, :]
    err = np.abs(g - p)
    vmin, vmax = min(g.min(), p.min()), max(g.max(), p.max())

    fig = plt.figure(figsize=(18, 5.5), dpi=120)
    plt.suptitle(f'{subject} — {ear} ear @ {f_actual:.0f} Hz    MAE {err.mean():.2f} dB',
                 fontsize=15, fontweight='bold', y=1.02)
    hp.mollview(g,   fig=fig.number, sub=(1, 3, 1), title='Ground truth',
                min=vmin, max=vmax, cmap='viridis', unit='dB', format='%.1f')
    hp.mollview(p,   fig=fig.number, sub=(1, 3, 2), title='Prediction',
                min=vmin, max=vmax, cmap='viridis', unit='dB', format='%.1f')
    hp.mollview(err, fig=fig.number, sub=(1, 3, 3), title='|error|',
                cmap='hot', unit='dB', format='%.1f')
    plt.tight_layout(); plt.show()


for hz in (1000, 8000, 12000):      # 低频 / notch 带 / 高频
    plot_healpix_map(gt_l, pred_l, hz, TARGET, 'Left')

# ---- 逐频率误差曲线：看清误差集中在哪个频段 ----
err_f = np.sqrt(((gt_l - pred_l) ** 2).mean(axis=1))
plt.figure(figsize=(11, 4), dpi=120)
plt.plot(freqs / 1000, err_f, lw=2)
plt.axvspan(6, 12, color='orange', alpha=.12, label='notch band')
plt.axvspan(0, config['band_lo_hz'] / 1000, color='gray', alpha=.12, label='out of loss band')
plt.xlabel('frequency (kHz)'); plt.ylabel('RMS error (dB)')
plt.title(f'{TARGET} left ear — error vs frequency')
plt.legend(); plt.grid(alpha=.3); plt.tight_layout(); plt.show()

## 训练后的验收测试

把 `analysis/latent_variance_decomp.py` 在新 checkpoint 上再跑一遍。如果镜像生效：

| | v1 (未镜像) | v2 期望 |
|---|---|---|
| `'哪只耳'` 方差占比 | 71.7% | **接近 0** |
| `'哪个人'` 方差占比 | 18.6% | **大幅上升** |

这是能直接证明改动生效的硬指标，也可以写进 thesis。

另外记得：**新数据分布与 v1 不同（raw 管线、218 人 vs 48 人），
旧的 2.14 / 2.33 dB 不可直接比较** —— 用 Cell 3 打印的 mean-HRTF 基线作参照。